# 10a-DINOv2 -- MTMC Stages 0-2: Frame Extraction + Detection + DINOv2 ReID Features

**DINOv2 ViT-L/14 pipeline. Run once. ~110-130 min on T4.**

| Stage | What | Time |
|---|---|---|
| 0 | Frame extraction (10 fps, 6 cameras) | ~20 min |
| 1 | YOLO detection + BotSort tracking | ~45 min |
| 2 | DINOv2 ViT-L/14 TransReID 1024D -> PCA 384D features | ~30 min |

After this runs, its output (`checkpoint.tar.gz`) is used by **10b** -> **10c**.
DINOv2 checkpoint is loaded from 09s kernel output (attached as kernel_source).

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _match = re.search(r"(\d+)\.(\d+)", _cap)
        if _match:
            _major, _minor = _match.groups()
            _sm = int(_major) * 10 + int(_minor)
            if _sm < 70:
                print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) — installing compatible PyTorch ...")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1", "torchvision==0.19.1",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])
                print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Keep Kaggle's preinstalled torch/torchvision/numpy stack intact.
pip("--no-deps", "ultralytics")
pip("filterpy", "ftfy", "lapx")
pip("--no-deps", "boxmot==11.0.3")

try:
    import torchreid; print("torchreid ok")
except ImportError:
    pip("git+https://github.com/KaiyangZhou/deep-person-reid.git")

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip("faiss-gpu")
    except Exception:
        pip("faiss-cpu")

pip("timm", "motmetrics")
pip("loguru", "omegaconf", "rich", "networkx>=3.1", "click")

# SAM2 - non-torch deps first, then SAM2 itself without deps
pip("hydra-core", "iopath")
pip("--no-deps", "git+https://github.com/facebookresearch/sam2.git")

# Install project in editable mode (no deps - all handled above)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
print("✓ All dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("ultralytics", "ultralytics"),
    ("boxmot", "boxmot"),
    ("torch", "torch"),
    ("torchreid", "torchreid"),
    ("timm", "timm"),
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("cv2", "cv2"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sam2", "sam2"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  ✓ {label}")
    except ImportError as exc:
        print(f"  ✗ {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("✓ All required modules importable")

## 2. Mount Model Weights
Model weights dataset (`mtmc-weights`) must be attached.

In [ ]:
# ── Download cross-account datasets that don't auto-mount ─────────────────
import subprocess, os
from pathlib import Path

def _download_if_missing(dataset_slug: str, expected_path: str):
    p = Path(expected_path)
    if p.exists():
        print(f"✓ {expected_path} already mounted")
        return
    name = dataset_slug.split("/")[-1]
    # Download to /tmp (always writable; /kaggle/input is read-only for writes)
    tmp_dir = Path(f"/tmp/{name}")
    tmp_dir.mkdir(parents=True, exist_ok=True)
    print(f"⬇ Downloading {dataset_slug} → {tmp_dir} ...")
    print(f"  KAGGLE_USERNAME={os.environ.get('KAGGLE_USERNAME', 'NOT SET')}")
    r = subprocess.run(
        ["kaggle", "datasets", "download", dataset_slug,
         "-p", str(tmp_dir), "--unzip"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(
            f"Download failed (rc={r.returncode}) for {dataset_slug}\n"
            f"STDOUT: {r.stdout[-800:]}\nSTDERR: {r.stderr[-800:]}"
        )
    print(f"✓ Downloaded to {tmp_dir}")
    print(r.stdout[-200:] if r.stdout else "")
    # Detect content root (zip may add a top-level folder)
    contents = list(tmp_dir.iterdir())
    content_root = contents[0] if len(contents) == 1 and contents[0].is_dir() else tmp_dir
    print(f"  Content root: {content_root}")
    print(f"  Contents: {[x.name for x in list(content_root.iterdir())[:8]]}")
    # Create symlink at expected path so downstream code finds it there
    try:
        p.parent.mkdir(parents=True, exist_ok=True)
        if not p.exists():
            os.symlink(str(content_root), str(p))
            print(f"✓ Symlinked {content_root} → {p}")
    except OSError as e:
        print(f"⚠ Could not symlink at {p}: {e}")
        print(f"  Weights available at: {content_root}")

_download_if_missing("gumfreddy/mtmc-weights", "/kaggle/input/mtmc-weights")

In [ ]:
WEIGHTS_INPUT = Path("/kaggle/input/mtmc-weights")
if not WEIGHTS_INPUT.exists():
    _fallback = Path("/tmp/mtmc-weights")
    if _fallback.exists():
        WEIGHTS_INPUT = _fallback
        print(f"Using fallback weights path: {WEIGHTS_INPUT}")

MODELS_DST = PROJECT / "models"
if MODELS_DST.is_symlink():
    MODELS_DST.unlink()
if MODELS_DST.exists():
    shutil.rmtree(MODELS_DST)
print(f"Copying models/ from {WEIGHTS_INPUT} (~750 MB) ...")
shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

# --- Debug: show what was copied ---
print("Contents of models/ after copy:")
for f in sorted(MODELS_DST.rglob("*"))[:30]:
    if f.is_file():
        print(f"  {f.relative_to(MODELS_DST)} ({f.stat().st_size/1024**2:.1f} MB)")

# --- Extract any zip files (--dir-mode zip uploads) ---
import zipfile
for zf in sorted(MODELS_DST.rglob("*.zip")):
    print(f"Extracting {zf.relative_to(MODELS_DST)} ...")
    with zipfile.ZipFile(zf) as z:
        z.extractall(zf.parent)
    zf.unlink()

# --- Handle flat structure: move root-level model files into subdirs ---
for subdir in ["detection", "reid", "tracker"]:
    (MODELS_DST / subdir).mkdir(exist_ok=True)

for f in list(MODELS_DST.glob("*.pt")):
    if "yolo" in f.name.lower():
        shutil.move(str(f), str(MODELS_DST / "detection" / f.name))
    elif "osnet" in f.name.lower():
        shutil.move(str(f), str(MODELS_DST / "tracker" / f.name))
for f in list(MODELS_DST.glob("*.pth")):
    shutil.move(str(f), str(MODELS_DST / "reid" / f.name))
for f in list(MODELS_DST.glob("*.pkl")):
    shutil.move(str(f), str(MODELS_DST / "reid" / f.name))
for f in list(MODELS_DST.glob("*.json")):
    if f.name != "dataset-metadata.json":
        shutil.move(str(f), str(MODELS_DST / "reid" / f.name))

print("Final models/ structure:")
for f in sorted(MODELS_DST.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(MODELS_DST)} ({f.stat().st_size/1024**2:.1f} MB)")

ESSENTIAL = [
    "models/detection/yolo26m.pt",
    "models/reid/transreid_cityflowv2_best.pth",
    "models/tracker/osnet_x0_25_msmt17.pt",
]
missing = [p for p in ESSENTIAL if not (PROJECT / p).exists()]
if missing:
    for missing_path in missing:
        print(f"  MISSING: {missing_path}")
    raise FileNotFoundError("Fix missing weights before continuing.")
print("✓ All essential baseline weights present")

In [ ]:
# --- Load DINOv2 ViT-L/14 checkpoint from 09s kernel output ---
import shutil
from pathlib import Path

DINOV2_CANDIDATES = [
    Path("/kaggle/input/notebooks/yahiaakhalafallah/09s-dinov2-large-cityflowv2/vehicle_transreid_dinov2_large_cityflowv2_final.pth"),
    Path("/kaggle/input/09s-dinov2-large-cityflowv2/vehicle_transreid_dinov2_large_cityflowv2_final.pth"),
]
DINOV2_DST = PROJECT / "models" / "reid" / "transreid_cityflowv2_best.pth"

dinov2_src = None
for c in DINOV2_CANDIDATES:
    if c.exists():
        dinov2_src = c
        break

if dinov2_src is None:
    # List all available kernel sources to help debug
    import os
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            p = os.path.join(root, f)
            if "dinov2" in f.lower() or "09s" in p.lower():
                print(f"Found: {p}")
    raise FileNotFoundError(
        "DINOv2 checkpoint not found! Check that yahiaakhalafallah/09s-dinov2-large-cityflowv2 is added as kernel_source"
    )

shutil.copy2(dinov2_src, DINOV2_DST)
print(f"✓ DINOv2 checkpoint copied: {dinov2_src} -> {DINOV2_DST}")
print(f"  Size: {DINOV2_DST.stat().st_size / 1024**2:.1f} MB")

Overwrite the baseline checkpoint path with the exported 09s DINOv2 ViT-L/14 weights.
Keep the 09p fine-tuned R50-IBN checkpoint available on disk as `vehicle2`, but disable it for this clean DINOv2 baseline run.
No `vehicle3` LAION branch is enabled in this notebook.

In [ ]:
# Keep vehicle2 assets available on disk, but the DINOv2 run disables vehicle2 in stage2.
import shutil
import urllib.request
from pathlib import Path

FASTREID_SECONDARY_PATH = PROJECT / "models" / "reid" / "fastreid_r50_ibn_cityflowv2.pth"

CKPT_ROOT_CANDIDATES = [
    Path("/kaggle/input/09p-r50-ibn-cityflowv2-extended-checkpoint"),
    Path("/kaggle/input/datasets/gumfreddy/09p-r50-ibn-cityflowv2-extended-checkpoint"),
    Path("/kaggle/input/09p-fastreid-r50-extended-cityflowv2"),
]

CKPT_FILE_CANDIDATES = [
    "fastreid_r50_ibn_cityflowv2_extended_final.pth"
    , "fastreid_r50_ibn_cityflowv2_extended_epoch_200.pth"
    , "checkpoints/fastreid_r50_ibn_cityflowv2_extended_final.pth"
    , "checkpoints/fastreid_r50_ibn_cityflowv2_extended_epoch_200.pth"
]

FALLBACK_09N = Path(
    "/kaggle/input/mtmc-10a-checkpoints/fastreid_r50_ibn_cityflowv2_final.pth"
)
FASTREID_SECONDARY_URL = (
    "https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth"
)


def resolve_secondary_ckpt() -> tuple[Path | None, str]:
    for root in CKPT_ROOT_CANDIDATES:
        for rel in CKPT_FILE_CANDIDATES:
            p = root / rel
            if p.exists():
                return p, f"09p:{p}"
    if FALLBACK_09N.exists():
        return FALLBACK_09N, f"09n:{FALLBACK_09N}"
    return None, "veri-url"


src_ckpt, src_tag = resolve_secondary_ckpt()
FASTREID_SECONDARY_PATH.parent.mkdir(parents=True, exist_ok=True)

if src_ckpt is not None:
    shutil.copy2(src_ckpt, FASTREID_SECONDARY_PATH)
    print(f"Copied secondary checkpoint ({src_tag}) -> {FASTREID_SECONDARY_PATH}")
else:
    urllib.request.urlretrieve(FASTREID_SECONDARY_URL, str(FASTREID_SECONDARY_PATH))
    print(f"Downloaded fallback VeRi checkpoint (veri-url) -> {FASTREID_SECONDARY_PATH}")

## 3. Mount CityFlowV2 Dataset

CityFlowV2 is attached as a Kaggle dataset. We symlink the nested
`train/S01/c001/` structure into the flat `S01_c001/` layout the pipeline expects.

In [ ]:
import re as _re, shutil as _shutil

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = _shutil.disk_usage(mount)
    print(f"{mount:20s}  {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

# --- Locate the Kaggle-mounted CityFlowV2 dataset ---
_CANDIDATE_MOUNTS = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = None
for _p in _CANDIDATE_MOUNTS:
    if _p.exists():
        CITYFLOW_INPUT = _p
        break
assert CITYFLOW_INPUT is not None, (
    "CityFlowV2 dataset not found.\n"
    "  Attach via: Add Data -> Datasets -> search 'data-aicity-2023-track-2'"
)
print(f"\u2713 CityFlowV2 dataset at {CITYFLOW_INPUT}")

# --- Flatten into data/raw/cityflowv2/S01_c001/ layout ---
TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)

DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists(): shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)
    print(f"\u2713 data/raw \u2192 {TMP_DATA}")
else:
    print(f"data/raw already symlinked -> {DATA_RAW_PARENT.resolve()}")

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)
DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Map: train/S01/c001 -> S01_c001, validation/S02/c006 -> S02_c006
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"  # e.g. S01_c001
            flat_dir = DATA_RAW / flat_name
            if flat_dir.exists():
                continue
            # Symlink the entire camera dir (read-only mount is fine for inference)
            flat_dir.symlink_to(cam_dir)

CAM_RE = _re.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(d.name for d in DATA_RAW.iterdir() if d.is_dir() and CAM_RE.match(d.name))
print(f"\u2713 CityFlowV2 ready: {len(cams)} cameras")
for c in cams: print(f"  {c}")
print(f"\nDataset path: {DATA_RAW}")

## 3b. Generate ROI Masks

Generate per-camera ROI masks from video using background subtraction.
These masks eliminate non-road detections (buildings, parked cars) and are
the single biggest factor in reducing false positives.

In [ ]:
os.chdir(str(PROJECT))
print("Generating ROI masks ...")
r = subprocess.run(
    [sys.executable, "scripts/generate_roi_masks.py",
     "--data-dir", str(DATA_RAW),
     "--n-samples", "200"],
    cwd=str(PROJECT))
if r.returncode != 0:
    print("WARNING: ROI mask generation failed (non-fatal, will proceed without masks)")
else:
    # Verify masks were created
    masks = list(DATA_RAW.glob("*/roi.jpg"))
    print(f"\u2713 Generated {len(masks)} ROI masks")

## 4. Run Stages 0-2

This cell runs the DINOv2 ViT-L/14 stage2 baseline with PCA 384D output.
The 09p R50-IBN checkpoint is staged on disk for comparison work, but disabled in this notebook's command overrides.

In [ ]:
from datetime import datetime
RUN_NAME = f"run_kaggle_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR  = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_CAMERAS = ['S01_c001', 'S01_c002', 'S01_c003', 'S02_c006', 'S02_c007', 'S02_c008']
print(f"Run  : {RUN_NAME}")
print(f"Cams : {BENCHMARK_CAMERAS}")

In [ ]:
os.chdir(str(PROJECT))
import os
import subprocess, time, sys

BASELINE_WEIGHTS = "models/reid/transreid_cityflowv2_best.pth"  # now DINOv2
SECONDARY_WEIGHTS = "models/reid/fastreid_r50_ibn_cityflowv2.pth"
# vehicle2 is DISABLED for clean DINOv2 baseline
print(f"Baseline vehicle model: {BASELINE_WEIGHTS} (exists={os.path.exists(BASELINE_WEIGHTS)})")
print(f"Secondary vehicle2 model: {SECONDARY_WEIGHTS} (exists={os.path.exists(SECONDARY_WEIGHTS)})")

cmd = [
    sys.executable, "scripts/run_pipeline.py"
    , "--config", "configs/default.yaml"
    , "--dataset-config", "configs/datasets/cityflowv2.yaml"
    , "--stages", "0,1,2"
    , "--override", f"project.run_name={RUN_NAME}"
    , "--override", f"project.output_dir={DATA_OUT}"
    , "--override", "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]"
    , "--override", "stage2.crop.samples_per_tracklet=48"
    , "--override", "stage2.crop.foreground_masking.enabled=false"
    , "--override", "stage2.pca.n_components=384"
    , "--override", "stage2.reid.color_augment=false"
    , "--override", "stage2.reid.vehicle.concat_patch=false"
    , "--override", "stage2.reid.vehicle.input_size=[252,252]"
    , "--override", "stage2.reid.vehicle.vit_model=vit_large_patch14_dinov2.lvd142m"
    , "--override", "stage2.reid.vehicle.clip_normalization=false"
    , "--override", "stage2.reid.vehicle.embedding_dim=1024"
    , "--override", "stage2.reid.vehicle2.enabled=false"
    , "--override", "stage2.multi_query.k=0"
    , "--override", "stage2.camera_bn.enabled=true"
    , "--override", "stage2.camera_tta.enabled=false"
    , "--override", "stage2.power_norm.alpha=0.0"
    , "--override", "stage1.interpolation.max_gap=50"
    , "--override", "stage1.intra_merge.max_time_gap=40.0"
    , "--override", "stage1.tracker.min_hits=2"
]

print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"Stage 0-2 done in {elapsed/60:.1f} min")

## 5. Save Checkpoint

Saves stage1 + stage2 outputs + GT annotations to `/kaggle/working/checkpoint.tar.gz`.
If `embeddings_secondary.npy` exists in `stage2`, it is included automatically.
This file becomes the input for **10b**.
Stage0 frame images (~9.6 GB) are **not** included -- downstream stages do not need them.

In [ ]:
    import re as _re2

    CAM_RE2 = _re2.compile(r"^S\d{2}_c\d{3}$")
    checkpoint_path = Path("/kaggle/working/checkpoint.tar.gz")
    metadata_path   = Path("/kaggle/working/run_metadata.json")
    secondary_embeddings_path = RUN_DIR / "stage2" / "embeddings_secondary.npy"

    with open(metadata_path, "w") as f:
        json.dump({"run_name": RUN_NAME}, f)

    print(f"Building checkpoint for run: {RUN_NAME}")
    if secondary_embeddings_path.exists():
        print(f"  ✓ Secondary embeddings detected: {secondary_embeddings_path}")
    else:
        print("  ⚠ Secondary embeddings not found; checkpoint will contain primary stage2 artifacts only")
    with tarfile.open(str(checkpoint_path), "w:gz") as tar:
        tar.add(str(metadata_path), arcname="run_metadata.json")

        manifest = RUN_DIR / "stage0" / "frames_manifest.json"
        if manifest.exists():
            tar.add(str(manifest), arcname=f"{RUN_NAME}/stage0/frames_manifest.json")
            print("  + stage0/frames_manifest.json")

        for stage in ["stage1", "stage2"]:
            stage_dir = RUN_DIR / stage
            if stage_dir.exists():
                n = 0
                for fpath in stage_dir.rglob("*"):
                    if fpath.is_file():
                        tar.add(str(fpath), arcname=f"{RUN_NAME}/{stage}/{fpath.relative_to(stage_dir)}")
                        n += 1
                print(f"  + {stage}/ ({n} files)")

        # GT annotation txt files needed by stage5 eval (small text files, not videos)
        gt_count = 0
        for cam_dir in DATA_RAW.iterdir():
            if cam_dir.is_dir() and CAM_RE2.match(cam_dir.name):
                gt_file = cam_dir / "gt" / "gt.txt"
                if gt_file.exists():
                    tar.add(str(gt_file), arcname=f"gt_annotations/{cam_dir.name}/gt/gt.txt")
                    gt_count += 1
        print(f"  + gt_annotations/ ({gt_count} GT files)")

    size_mb = checkpoint_path.stat().st_size / 1024**2
    print(f"\n✓ Checkpoint: {checkpoint_path}  ({size_mb:.1f} MB)")
    print("  Next: attach this notebook's output to 10b, then push 10b.")